# RAG System Observability Notebook

This notebook demonstrates and inspects each component of the Wikipedia RAG system.

## Pipeline Overview
```
Wikipedia Data → Embeddings → Vector Storage → Similarity Search → RAG Generation
```

In [ ]:
# Setup
import sys
from pathlib import Path

# Add src to path
src_path = Path('../src')
sys.path.insert(0, str(src_path))

import pandas as pd
import numpy as np
from data_loader import WikipediaDataLoader
from embedding_generator import EmbeddingGenerator
from vector_store import RAGVectorStore

print("RAG System Components Loaded")

## 1. Data Loading Inspection

In [ ]:
# Load and inspect Wikipedia data
loader = WikipediaDataLoader()
passages_df = loader.load_passages(limit=10)

print(f"Loaded: {len(passages_df)} passages")
print(f"Columns: {list(passages_df.columns)}")

# Display sample data
passages_df.head()

In [ ]:
# Analyze text lengths
passages_df['text_length'] = passages_df['passage'].str.len()

print("Text Length Statistics:")
print(passages_df['text_length'].describe())

# Convert to documents
documents = loader.passages_to_documents(passages_df)
print(f"\nCreated {len(documents)} LlamaIndex Documents")

## 2. Embedding Generation Inspection

In [ ]:
# Generate embeddings for first 3 documents
generator = EmbeddingGenerator(chunk_size=400, chunk_overlap=50)

small_docs = documents[:3]
embedded_nodes, stats = generator.generate_embeddings_for_documents(small_docs, batch_size=1)

print(f"Embedding Statistics:")
print(f"  Documents: {stats.total_documents}")
print(f"  Nodes: {stats.total_nodes}")
print(f"  Dimension: {stats.average_embedding_dimension}")
print(f"  Processing time: {stats.processing_time_seconds:.1f}s")
print(f"  Success rate: {(stats.total_nodes - stats.failed_embeddings) / stats.total_nodes:.1%}")

In [ ]:
# Inspect embedding vectors
first_embedded = next(node for node in embedded_nodes if hasattr(node, 'embedding') and node.embedding)

print(f"First embedding sample:")
print(f"  Text: {first_embedded.text[:100]}...")
print(f"  Vector length: {len(first_embedded.embedding)}")
print(f"  Sample values: {first_embedded.embedding[:5]}")
print(f"  Value range: [{min(first_embedded.embedding):.3f}, {max(first_embedded.embedding):.3f}]")

## 3. Vector Storage Inspection

In [ ]:
# Create vector store
vector_store = RAGVectorStore(storage_dir="../notebook_vector_storage", index_name="demo_index")
index = vector_store.create_index_from_nodes(embedded_nodes)

# Get index information
info = vector_store.get_index_info()
print("Vector Store Info:")
for key, value in info.items():
    print(f"  {key}: {value}")

## 4. Similarity Search Analysis

In [ ]:
# Test different queries
test_queries = [
    "Uruguay capital",
    "South America", 
    "Montevideo",
    "population statistics"
]

query_results = {}

for query in test_queries:
    results = vector_store.query_similar_documents(query, top_k=2)
    query_results[query] = results
    
    print(f"\nQuery: '{query}'")
    for i, result in enumerate(results, 1):
        score = result.get('similarity_score', 0)
        text_preview = result['text'][:80] + "..."
        print(f"  {i}. Score: {score:.3f} | {text_preview}")

## 5. RAG Generation Testing

In [ ]:
# Create query engine for RAG
query_engine = vector_store.get_query_engine(similarity_top_k=2)

# Test RAG queries
rag_queries = [
    "What is Uruguay?",
    "What is the capital of Uruguay?"
]

for query in rag_queries:
    print(f"\nRAG Query: '{query}'")
    try:
        response = query_engine.query(query)
        print(f"Answer: {str(response)}")
        
        if hasattr(response, 'source_nodes') and response.source_nodes:
            print(f"Sources used: {len(response.source_nodes)}")
            for i, node in enumerate(response.source_nodes[:2], 1):
                score = getattr(node, 'score', 'N/A')
                print(f"  Source {i}: Score={score} | {node.text[:60]}...")
    except Exception as e:
        print(f"Error: {e}")

## 6. Pipeline Performance Summary

In [ ]:
# Save vector store for persistence test
save_success = vector_store.save_index(overwrite=True)

# Summary statistics
print("RAG Pipeline Performance Summary:")
print("=" * 40)
print(f"✅ Data Loading: {len(passages_df)} passages loaded")
print(f"✅ Embedding Generation: {stats.total_nodes} nodes, {stats.average_embedding_dimension}D vectors")
print(f"✅ Vector Storage: Index created and {'saved' if save_success else 'not saved'}")
print(f"✅ Similarity Search: {len(test_queries)} queries tested")
print(f"✅ RAG Generation: Query engine working")

print(f"\nPerformance Metrics:")
print(f"  Embedding speed: {stats.embeddings_per_second:.1f} embeddings/sec")
print(f"  Processing time: {stats.processing_time_seconds:.1f} seconds")
print(f"  Success rate: {((stats.total_nodes - stats.failed_embeddings) / stats.total_nodes * 100):.1f}%")

print(f"\n🎉 RAG System Fully Operational!")